# MAKRO 2026: a simple experiment with standard ModelFlow

Public purchases of goods and services up by 1 per cent, permanently from 2030. That is about 0.1 per cent of
GDP, small enough to solve in one step.

Only standard ModelFlow is used: `modelload`, the model call with `keep=`, `.upd`, `fix`, and the standard
tables and charts. `makro26_experiment.ipynb` does more (DREAM's standard shocks, the two-step solve for
large shocks, comparison with DREAM).

**What it needs:** ModelFlow from `C:\modelflow2\modelflow` (the `factorizer=` option), `pypardiso`, and about
25 GB of free memory. Each solve of the whole horizon takes 1-3 minutes.

In [1]:
from modelclass import model
model.plain_html = True   # results as plain html tabs, so they also show in the book

## 1. Load

The file holds the equations, the baseline 2015-2131, the descriptions, the variable groups and the derivative
expressions. `straight=True` skips the analysis of the simultaneous structure (2 minutes), which the stacked
Newton does not use.

In [2]:
%%time
mmakro, baseline = model.modelload(r'C:\mfmodeller_raw\makro\makro26\data\makro26_baseline.pcimz',
                                   run=0, straight=True)

Feather/zip file read: C:\mfmodeller_raw\makro\makro26\data\makro26_baseline.pcimz


Model: MAKRO 2026-September (DREAM) in ModelFlow, baseline
CPU times: total: 1.69 s
Wall time: 1.37 s


## 2. Run the baseline

MAKRO looks forward, so all years are solved together (`newtonstack_ng`, ModelFlow's default for this model).
The span must end in the terminal year 2129.

| Option | Why |
| --- | --- |
| `forcenum=False` | exact derivatives; numerical ones are not accurate enough for MAKRO |
| `factorizer='normal'` | the linear system (1.5 million unknowns) through the normal equations and PARDISO |
| `keep='Baseline'` | keep the solution for the charts |
| `setbase=True` | the solution becomes the base that later results are compared with |

ModelFlow remembers the options, so the experiment below uses them too. The baseline data already solves the
model, so the run should take one iteration and change nothing.

In [3]:
%%time
first, last = 2030, 2129
base = mmakro(baseline, first, last, solver='newtonstack_ng', forcenum=False, factorizer='normal',
              silent=0, keep='Baseline', setbase=True)

Will start solving: MAKRO26
Building the stacked pro/core/epi split


Stacked split: 15,472 core / 2,834 prolog / 2,168 epilog variables
makelos_ng compiles the stacked split evaluators for MAKRO26


Prepare model for calculate derivatives for Newton solver
The expressions for the derivatives are taken from the model (diff_expressions)


Calculating new derivatives and create new stacked Newton solver
Will start solving: MAKRO26 Derivatives  all lags and leads
New data or transpile_reset
Create compiled solving function for MAKRO26 Derivatives  all lags and leads
now makelos makes a res solvefunction


MAKRO26 Derivatives  all lags and leads solved  


normal: 1,547,200 unknowns, 7,653,248 non-zeros, 50.7 s, J'J 85,669,778 non-zeros


Iteration  0 Max residual            0.000000004411


   normal solve: relative linear residual 4.6e-12 after 4 refinements


Solved in 0 iterations
MAKRO26 solved  
CPU times: total: 10min 7s
Wall time: 1min 8s


## 3. The experiment

`QR__OFF` is public purchases of goods and services, `qR['off',t]` in MAKRO. The update line raises it by
1 per cent in all years.

In [4]:
mmakro['QR__OFF'].des

QR__OFF : Materialeinput fordelt på brancherne som modtager materialeinputtet, Kilde: ADAM[fVm] eller ADAM[fVm<i>] [OFF] (MAKRO qR)


In [5]:
alt = base.upd(f'<{first} {last}> QR__OFF * 1.01')

**The first year.** A shock is unanticipated. GAMS leaves a few equations out in the first year of a run: the
required return on shares, the firms' discount rate and factor and their risk premia (39 variables). Their
values for that year were set before the shock. `fix` does the same here: it fixes these variables at their
baseline value in 2030 (sets `_D` = 1 and `_X` = the value).

In [6]:
alt = mmakro.fix(alt, 'RAKTIEDRIFT* RVIRKDISK* FVIRKDISK* RFINACCELPREM* DFINFRIKTION* SFVIRKDISK* SDOPSLAGOMK2DNLLAG*',
                 first, first, silent=1)

## 4. Solve the experiment

One step, with the options from the baseline run.

In [7]:
%%time
res = mmakro(alt, first, last, keep='Public purchases +1%')

Will start solving: MAKRO26


Iteration  0 Max residual            3.636522453038


   normal solve: relative linear residual 2.3e-12 after 6 refinements


Iteration  1 Max residual            0.067403924964


   normal solve: relative linear residual 5.9e-12 after 5 refinements


Iteration  2 Max residual            0.060389731365


... (6 outputs left out)

   normal solve: relative linear residual 1.7e-12 after 3 refinements


Iteration  6 Max residual            0.000000603405


   normal solve: relative linear residual 6.3e-13 after 3 refinements


Solved in 6 iterations
MAKRO26 solved  
CPU times: total: 8min 13s
Wall time: 57.6 s


## 5. The impact

`mmakro[...]` compares the latest solution with the base. `#Headline` is a variable group stored with the
model. Quantities (`Q...`) and prices (`P...`) are growth- and inflation-adjusted in MAKRO; a percentage
difference from the baseline is the same in adjusted and ordinary units.

In [8]:
mmakro.set_smpl(first, first + 30)
mmakro['#Headline']

## Or results in levels 

In [9]:
mmakro['#Headline'].rplot(datatype='level') 

## And the impact of the experiment in percent of the level
Other options are will show growth rates, impact on growth rate and so on 

In [10]:
mmakro['#Headline'].rplot(datatype='difpctlevel') 

## In a live jupyter notebook a interactive widget can be used 

In [11]:
mmakro.keep_plot_widget() 

## 6. What the model instance holds

`tools\model_info.py` reads only what is stored on the instance (counts, data, kept solutions, the
Jacobian the last solve left), for the time frame `mmakro.current_per` - after the solve the whole
simulation period 2030-2129. 

In [12]:
import sys
sys.path.insert(0, r'C:\mfmodeller_raw\makro\makro26\tools')
from model_info import model_info, jacobian_info

J1, JS = model_info(mmakro)

variables                                39,034
equations (endogenous)                   20,474
  normalized  X = f(...)                 20,100
  implicit    X___RES = f(...)           374
exogenous                                18,560
  solved through an implicit equation    374
  _D/_X fix switches                     3,304
  other inputs                           14,882
max lag / max lead                       -4 / 2
loaded straight (no structure analysis)  yes
time frame (current_per)                 2030-2129 (100)
lastdf                                   117 x 39,034 (2015-2131, 37 MB)
basedf                                   117 x 39,034 (2015-2131, 37 MB)
kept solutions                           Baseline, Public purchases +1%
stacked solve split pro / core / epi     2,834 / 15,472 / 2,168
stored derivative expressions            1 set(s)
last solver                              newtonstack_ng

time frame (current_per)                                   2030-2129 (100)
stacked J